# NLP Lab : Week 4 (Exercises)
## Module 9: Comparing BERT, GPT-2, RoBERTa, and DistilBERT for Body Performance Classification

## Important Note

This notebook follows the same structure as the starter but uses the **Body Performance** dataset :
a four-class problem. Fill in every `??` placeholder to complete the notebook.

### Key Changes from Starter Notebook

| Setting | Starter (Diabetes) | Exercises (Body Performance) |
|---------|-------------------|------------------------------|
| Dataset | diabetes.csv : binary (0 / 1) | bodyPerformance.csv : 4 classes (A / B / C / D) |
| `NUM_CLASSES` | 2 (hard-coded) | `len(set(y_train))` |
| `num_labels` | 2 | `NUM_CLASSES` |
| Loss | `CrossEntropyLoss` (weighted) | `CrossEntropyLoss` |
| Precision / Recall / F1 | `average='binary'` (default) | `average='weighted'` |
| ROC-AUC | `roc_auc_score(y, proba[:, 1])` : binary | `roc_auc_score(y, proba, multi_class='ovr', average='weighted')` |
| Confusion matrix labels | `['No Diabetes', 'Diabetes']` | `['A', 'B', 'C', 'D']` |
| Sort results by | Accuracy | F1 |


### Section 0: Set-up and Libraries
> Have a look at the new libraries compared with Week 3.

In [ ]:
# Standard libraries
import time
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
)

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# HuggingFace Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Hyperparameters
MAX_LENGTH = 128   # body performance text strings are longer (11 features)
BATCH_SIZE = 16
NUM_EPOCHS = 3
LR         = 2e-5


### Section 1: Load and Inspect the Dataset
> Download the body performance CSV from this [Kaggle Dataset](https://www.kaggle.com/datasets/kukuroo3/body-performance-data) and place it in the same directory as this notebook.

Expected file: `bodyPerformance.csv`

The dataset contains 11 features:
age, gender, height\_cm, weight\_kg, body fat %, diastolic, systolic,
gripForce, sit and bend forward\_cm, sit-ups counts, broad jump\_cm

**Target Variable:** `class` : A (best), B, C, D (weakest)


In [ ]:
# Load the dataset
df = pd.read_csv('bodyPerformance.csv')

print('Shape:', df.shape)
display(df.head())
df.info()

# Separate features and target
feature_cols = [c for c in df.columns if c != 'class']
X_df = df[feature_cols]

# Encode class labels: A→0, B→1, C→2, D→3
le = LabelEncoder()
y  = le.fit_transform(df['class'])   # integer array
CLASS_NAMES = le.classes_.tolist()   # ['A', 'B', 'C', 'D']

print('\nClass mapping:', dict(zip(CLASS_NAMES, le.transform(CLASS_NAMES))))
print('Class distribution:')
print(pd.Series(df['class']).value_counts().sort_index())

# Train / test split (use a 2000 / 500 subset for practical runtimes;
# remove .iloc[:2500] to train on the full dataset)
df_sub = df.sample(n=2500, random_state=42)
X_sub  = df_sub[feature_cols]
y_sub  = le.transform(df_sub['class'])

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_sub, y_sub,
    test_size=0.2, random_state=42, stratify=y_sub
)

# Derive number of classes from training labels
NUM_CLASSES = ??

print(f'\nTraining samples : {len(y_train)}')
print(f'Test samples     : {len(y_test)}')
print(f'Number of classes: {NUM_CLASSES}')
print(f'Class names      : {CLASS_NAMES}')


### Section 2: Text Serialisation, Dataset Class, and DataLoader Helper

Each row is converted into a descriptive text string before tokenisation.
Complete the `serialise_row` function below.


In [ ]:
def serialise_row(row: pd.Series) -> str:
    """
    Convert a single tabular row into a descriptive text string.
    Integer-valued features are printed without a decimal point;
    float-valued features are rounded to two decimal places.
    """
    parts = []
    for col, val in row.items():
        if isinstance(val, float) and val != int(val):
            parts.append(f'{col}: {val:.2f}')
        else:
            parts.append(f'{col}: {val}')  # gender 'M'/'F' is a string
    return ', '.join(parts)


train_texts = [serialise_row(row) for _, row in X_train_df.iterrows()]
test_texts  = [serialise_row(row) for _, row in X_test_df.iterrows()]

print('Example serialised row:')
print(train_texts[0])
print('\nLabel:', y_train[0], '→', CLASS_NAMES[y_train[0]])


In [ ]:
class TextDataset(Dataset):
    """Wraps tokenised text encodings and integer labels for PyTorch."""
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def create_dataloaders(tokenizer, train_texts, y_train,
                       test_texts, y_test,
                       max_length=MAX_LENGTH, batch_size=BATCH_SIZE):
    """Tokenise the train and test text lists and return (train_loader, test_loader)."""
    train_enc = tokenizer(
        train_texts, truncation=True, padding=True, max_length=max_length
    )
    test_enc  = tokenizer(
        test_texts,  truncation=True, padding=True, max_length=max_length
    )
    train_loader = DataLoader(
        TextDataset(train_enc, list(y_train)),
        batch_size=batch_size, shuffle=True
    )
    test_loader  = DataLoader(
        TextDataset(test_enc,  list(y_test)),
        batch_size=batch_size, shuffle=False
    )
    return train_loader, test_loader


### Section 3: Tokenisers and DataLoaders

GPT-2 has no dedicated padding token : complete the fix below.


In [ ]:
# ── BERT ──────────────────────────────────────────────────────────────────
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_train_loader, bert_test_loader = create_dataloaders(
    bert_tokenizer, train_texts, y_train, test_texts, y_test
)

# ── GPT-2 ─────────────────────────────────────────────────────────────────
gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')
# GPT-2 has no pad token : assign the EOS token as the pad token
gpt2_tokenizer.pad_token = ??
gpt2_train_loader, gpt2_test_loader = create_dataloaders(
    gpt2_tokenizer, train_texts, y_train, test_texts, y_test
)

# ── RoBERTa ───────────────────────────────────────────────────────────────
roberta_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
roberta_train_loader, roberta_test_loader = create_dataloaders(
    roberta_tokenizer, train_texts, y_train, test_texts, y_test
)

# ── DistilBERT ────────────────────────────────────────────────────────────
distilbert_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
distilbert_train_loader, distilbert_test_loader = create_dataloaders(
    distilbert_tokenizer, train_texts, y_train, test_texts, y_test
)

print('All tokenisers and DataLoaders created.')


### Section 4: Model 1 : BERT

12 encoder layers, ~110 M parameters, bidirectional encoder.

Set `num_labels` to the number of body performance classes.


In [ ]:
logging.set_verbosity_error()
bert_model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
     num_labels = ??,
).to(device)

print(bert_model.config)
print(f'\nTotal parameters: {sum(p.numel() for p in bert_model.parameters()):,}')


### Section 5: Model 2 : GPT-2

12 decoder layers, ~117 M parameters, unidirectional decoder.

Set `num_labels` to the number of body performance classes.


In [ ]:
logging.set_verbosity_error()
gpt2_model = AutoModelForSequenceClassification.from_pretrained(
    'gpt2',
    num_labels = ??,
).to(device)

# Mirror the tokeniser pad-token fix in the model config
gpt2_model.config.pad_token_id = ??

print(gpt2_model.config)
print(f'\nTotal parameters: {sum(p.numel() for p in gpt2_model.parameters()):,}')


### Section 6: Model 3 : RoBERTa

12 encoder layers, ~125 M parameters, optimised BERT pre-training.

Set `num_labels` to the number of body performance classes.


In [ ]:
logging.set_verbosity_error()
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels = ??,
).to(device)

print(roberta_model.config)
print(f'\nTotal parameters: {sum(p.numel() for p in roberta_model.parameters()):,}')


### Section 7: Model 4 : DistilBERT

6 encoder layers, ~66 M parameters, distilled from BERT.

Set `num_labels` to the number of body performance classes.


In [ ]:
logging.set_verbosity_error()
distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
     num_labels = ??,
).to(device)

print(distilbert_model.config)
print(f'\nTotal parameters: {sum(p.numel() for p in distilbert_model.parameters()):,}')


### Section 8: Training Function

Complete the loss function. `CrossEntropyLoss` works for **any** number of classes ≥ 2.


In [ ]:
def train_model(model, train_loader, num_epochs=NUM_EPOCHS, lr=LR):
    """
    Fine-tune a HuggingFace sequence-classification model.
    Returns (losses, training_time).
    """
    # CrossEntropyLoss handles 2 or more classes uniformly
    criterion = ??

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    model.train()
    losses = []
    start_time = time.time()

    for epoch in range(num_epochs):
        epoch_loss = 0.0

        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        print(f'  Epoch {epoch + 1}/{num_epochs}  loss={avg_loss:.4f}')

    return losses, time.time() - start_time


### Section 9: Evaluation and Confusion Matrix Functions

Complete the multiclass evaluation placeholders:
- **Softmax** converts logits to per-class probabilities (shape N × 4).
- **Argmax** selects the predicted class label.
- **`average='weighted'`** weights each class by support (number of true instances),
  which is more informative than `'macro'` when classes may be unbalanced.
- **`multi_class='ovr'`** computes ROC-AUC for each class against all others (One-vs-Rest),
  then averages the per-class AUC scores with weighted averaging.


In [ ]:
def evaluate_model(model, test_loader):
    """
    Evaluate a fine-tuned transformer on the test DataLoader.
    Returns a dict with Accuracy, Precision, Recall, F1, ROC_AUC.
    """
    model.eval()
    all_logits, all_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            all_logits.append(outputs.logits.cpu())
            all_labels.append(labels.cpu())

    all_logits = torch.cat(all_logits, dim=0)   # shape: (N, NUM_CLASSES)
    all_labels = torch.cat(all_labels, dim=0).numpy()

    # Softmax → probability matrix of shape (N, NUM_CLASSES)
    probabilities = ??

    # Predicted class: index with the highest logit
    predictions   = ??

    return {
        'Accuracy' : accuracy_score( all_labels, predictions),
        # weighted average accounts for class-size differences
        'Precision': precision_score(all_labels, predictions, average=??, zero_division=0), 
        'Recall'   : recall_score(   all_labels, predictions, average=??, zero_division=0),  
        'F1'       : f1_score(       all_labels, predictions, average=??, zero_division=0),  
        # One-vs-Rest multiclass ROC-AUC
        'ROC_AUC'  : roc_auc_score(
            all_labels, probabilities,
            multi_class = ??,
            average='weighted'
        ),
    }


def plot_confusion_matrix(model, test_loader, name):
    """Plot a 4×4 confusion matrix with body performance class names."""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    fig, ax = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay.from_predictions(
        all_labels, all_preds,
        display_labels=['A', 'B', 'C', 'D'],
        colorbar=False, ax=ax
    )
    ax.set_title(f'{name}: Confusion Matrix')
    plt.tight_layout()
    plt.show()


### Section 10: Training all Models
> Training four large pre-trained models will take significant time on CPU.
> Use a GPU runtime (e.g. Google Colab with a T4) for faster execution.

In [ ]:
results = []

models = {
    'BERT'      : (bert_model,       bert_train_loader,       bert_test_loader),
    'GPT-2'     : (gpt2_model,       gpt2_train_loader,       gpt2_test_loader),
    'RoBERTa'   : (roberta_model,    roberta_train_loader,    roberta_test_loader),
    'DistilBERT': (distilbert_model, distilbert_train_loader, distilbert_test_loader),
}

for name, (model, train_loader, test_loader) in models.items():

    print(f'\n{"="*50}')
    print(f' Training {name}')
    print(f'{"="*50}')

    losses, train_time = train_model(model, train_loader)

    metrics = evaluate_model(model, test_loader)
    metrics['Model']              = name
    metrics['Training Time (s)'] = round(train_time, 2)
    results.append(metrics)

    # Training loss curve
    plt.figure(figsize=(6, 4))
    plt.plot(losses, marker='o')
    plt.title(f'{name} Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.xticks(range(NUM_EPOCHS), range(1, NUM_EPOCHS + 1))
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    # Confusion matrix
    plot_confusion_matrix(model, test_loader, name)


### Section 11: Result Analysis of all Models
> Code for results analysis

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df[
    ['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'Training Time (s)']
]

# Sort by F1 (weighted) : more meaningful than Accuracy for multi-class problems
# because it captures both precision and recall across all four classes.
results_df.sort_values(by='F1', ascending=False)


## Reflection Questions

1. `CrossEntropyLoss` was also used in the starter notebook for binary classification. Explain why the same loss function works for both 2-class and 4-class problems. How does it differ mathematically from `BCEWithLogitsLoss`?
2. Why do we use `average='weighted'` rather than `average='macro'` for precision, recall, and F1 in this notebook? In which scenario would `'macro'` be the better choice?
3. Explain the `multi_class='ovr'` strategy used in `roc_auc_score`. What AUC score is computed for each class, and how are the four scores combined into a single number?


## Extension Exercises

Try the following modifications and note their effect on F1.

1. **Full dataset** : Remove the `.sample(n=2500)` subset and retrain on all ~13 000 samples. Does F1 improve significantly?
2. **Epochs** : Re-train the best model with `NUM_EPOCHS = 5`. Does additional training help or cause overfitting?
3. **Sequence length** : Change `MAX_LENGTH` from 128 to 64. What is the trade-off between speed and accuracy?
4. **Learning rate** : Try `lr=5e-5`. How does a higher learning rate affect convergence and final F1?

> Adjust your code and explain what you observe after each modification, along with possible reasons for those observations.
